In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os
import tensorflow as tf
from PIL import Image

digits = pd.read_csv("../mnist_combined.csv")
letters = pd.read_csv("../emnist_combined_letters.csv")



In [18]:
def fix_emnist_letters_orientation(X_letters):
    X_letters = X_letters.reshape(-1, 28, 28)
    fixed_images = []
    for img in X_letters:
        pil_img = Image.fromarray(img.astype(np.uint8))
        pil_img = pil_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        pil_img = pil_img.rotate(90, expand=False)
        fixed_images.append(np.array(pil_img))
    return np.array(fixed_images).reshape(-1, 784)


In [19]:
X_digits = digits.iloc[:, 1:].values.astype("float32")
y_digits = digits.iloc[:, 0].values.astype("int64")

X_letters = fix_emnist_letters_orientation(X_letters)
y_letters = letters.iloc[:, 0].values.astype("int64")


In [20]:
y_letters = y_letters - 1
y_letters = y_letters + 10

In [21]:
X = np.vstack([X_digits, X_letters])
y = np.concatenate([y_digits, y_letters])


X = X / 255.0



In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes=36)
y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=36)

In [24]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(36, activation="softmax")
])

In [25]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 570,788 (2.18 MB)

 Trainable params: 570,788 (2.18 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [27]:

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_test, y_test_onehot),
    epochs=40,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7338 - loss: 0.9039 - val_accuracy: 0.8824 - val_loss: 0.3813
Epoch 2/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8550 - loss: 0.4664 - val_accuracy: 0.9041 - val_loss: 0.3016
Epoch 3/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8815 - loss: 0.3806 - val_accuracy: 0.9170 - val_loss: 0.2635
Epoch 4/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8943 - loss: 0.3369 - val_accuracy: 0.9197 - val_loss: 0.2521
Epoch 5/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9013 - loss: 0.3081 - val_accuracy: 0.9280 - val_loss: 0.2279
Epoch 6/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9092 - loss: 0.2854 - val_accuracy: 0.9276 - val_loss: 0.2229
Epoch 7/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9123 - loss: 0.2705 - val_accuracy: 0.9310 - val_loss: 0.2173
Epoch 8/40
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9159 - loss: 0.2595 - 

In [28]:
model.save("../trained_models/ann_model.keras")

print("Model saved!")

Model saved!
